# PA-06 · ¿Es válida esta comparación?

**Una cifra bien calculada no prueba que lo que se compara sea comparable** · Tráfico aéreo internacional de Australia (BITRE), enero de 2009 – noviembre de 2025

Clase 5 · Unidad 3 — Calidad de datos y análisis exploratorio

| | |
|---|---|
| **Núcleo de clase** | ≈ 27 min entre trabajo y puesta en común (partes 0 a 6). Reparto orientativo: lectura 2 · inspección y primera impresión 3,5 · dónde cambia la serie 1,5 · documentación 4,5 · decidir y defender 7,5 · verificar y recalcular 2,5 · balance 0,5 · puesta en común 5 |
| **Para casa** | ≤ 20 min (parte 7) |
| **Qué entregas** | Este mismo notebook, con tus etiquetas, tus citas de la documentación y tus defensas completadas |
| **Competencia** | ENTENDER → DECIDIR → IMPLEMENTAR → VALIDAR → DEFENDER |

## El problema

La dirección de una aerolínea consultora presenta cinco afirmaciones sobre el tráfico aéreo internacional de Australia. **En las cinco, la aritmética es correcta**: cada cifra sale de la tabla y se puede reproducir. Tu trabajo no es repetir el cálculo, sino decidir **si cada afirmación puede mantenerse tal como está**.

Para cada afirmación vas a elegir **una de cuatro etiquetas** y a **defenderla con una línea concreta de la documentación** del libro de origen:

| Etiqueta | Qué significa |
|---|---|
| `ACCEPT` | La afirmación se sostiene tal como está: nada de lo documentado afecta a esta comparación con esta tabla. |
| `QUALIFY` | La cifra sirve, pero la afirmación hay que reformularla o acotarla: decir qué mide y qué no mide. |
| `RECOMPUTE` | La documentación da una regla con la que se puede reconstruir, **a partir de esta misma tabla**, una cifra comparable: hay que recalcular y volver a comparar. |
| `DO NOT COMPARE` | Los dos lados de la comparación no están en la misma base y la documentación no da forma de ponerlos en la misma: la comparación no debe hacerse. |

Lo que **no** se hace aquí: predecir, descomponer series por estacionalidad, buscar causas ni medir significación estadística, ni saber de aviación más allá del glosario. Python es un instrumento para obtener evidencia; lo que se trabaja es **cómo se produjo la serie** y **si lo documentado afecta a lo que se compara**.

## Qué te damos

| Elemento | Qué es |
|---|---|
| `../data/international_airline_activity_2009-01_2025-11.csv` | La tabla: una fila por **mes × aerolínea × país del puerto** |
| `../data/figuras/` | Un gráfico por afirmación, tal como lo vio la dirección |
| `../data/docs/EXTRACTO_DOCUMENTACION.md` | **Extracto de la documentación del libro de origen** (copia literal en inglés, con un ID por bloque) |
| `../data/DATASET_NOTICE.md` | Procedencia y qué se ha tocado y qué no |
| Funciones ya escritas | Series por aerolínea, ventanas de meses, lectura del extracto y comprobaciones |

> **Código proporcionado — no se evalúa esta sintaxis.** Las celdas usan `pandas` (`groupby`, `reindex`, filtros). No tienes que escribirlas ni memorizarlas: **escribes etiquetas, IDs, palabras, meses, listas y frases cortas**. Sí eres responsable de **entender qué consigue el código, si el resultado es válido y qué significa**.

### GLOSARIO ELABORADO PARA ESTA ACTIVIDAD

La documentación original está en inglés; este glosario **no es documentación original del conjunto de datos**: lo hemos elaborado solo con lo que necesitas para leer la actividad.

| Término | En esta actividad |
|---|---|
| *tonne* (tonelada) | 1.000 kg. La carga (`Freight`) y el correo (`Mail`) se miden en toneladas |
| *uplift / discharge* | Pasajeros o carga **embarcados** (uplift) o **desembarcados** (discharge) en un puerto |
| *revenue passenger* | Pasajero que ha pagado billete (se contabiliza en las estadísticas) |
| *gross / net weight* | Peso **bruto**: incluye embalajes, palés y contenedores. Peso **neto**: solo la mercancía |
| *year ended …* | Periodo de 12 meses que **termina** en el mes que se indica |

## Cómo se trabaja: DECLARA, LUEGO VERIFICA

1. **Declaras** (una primera impresión, una etiqueta, una lectura de una columna) *antes* de ver la comprobación.
2. **Verificas** con una comprobación ejecutable.
3. Si el resultado te sorprende, esa sorpresa es lo valioso: apunta por qué.

## Sobre el uso de IA

Puedes usar documentación externa y herramientas de IA. **La IA puede generar; tú sigues siendo responsable** de entender, comprobar, poner a prueba y defender lo que entregas. Si tu conclusión depende de algo que **no está** en el extracto ni en la tabla, indícalo y distingue qué demuestran tus datos de qué añade esa fuente. Un código que se ejecuta y da una cifra verosímil no está por eso validado.

---
## Parte 0 · Preparación

Ejecuta las celdas. No hay nada que decidir aquí.

In [ ]:
# SOLO PARA GOOGLE COLAB (añadido para el repositorio; en Jupyter o VS Code esta celda no hace nada).
# Descarga desde GitHub la carpeta data/ de esta práctica y se sitúa en student/,
# para que las rutas "../data/..." del notebook funcionen igual que en local.
import os, sys
if "google.colab" in sys.modules:
    import urllib.parse, urllib.request
    BASE = "https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/BigData/PA-06/"
    FICHEROS = [
        'data/docs/EXTRACTO_DOCUMENTACION.md',
        'data/international_airline_activity_2009-01_2025-11.csv',
    ]
    for f in FICHEROS:
        destino = os.path.join("/content/PA-06", f)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        if not os.path.exists(destino):
            urllib.request.urlretrieve(BASE + urllib.parse.quote(f), destino)
    os.makedirs("/content/PA-06/student", exist_ok=True)
    os.chdir("/content/PA-06/student")
    print(f"Colab: {len(FICHEROS)} ficheros de datos listos en /content/PA-06/data")

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve y qué significa el resultado)

import math

import pandas as pd
from IPython.display import Markdown, display

RUTA = "../data/international_airline_activity_2009-01_2025-11.csv"
RUTA_EXTRACTO = "../data/docs/EXTRACTO_DOCUMENTACION.md"
ETIQUETAS = ["ACCEPT", "QUALIFY", "RECOMPUTE", "DO NOT COMPARE"]


def comprobar(condicion, texto):
    """Como un assert, pero no detiene el notebook: imprime ✔ si se cumple y ✘ si no."""
    print(("✔ " if condicion else "✘ ") + texto)


print("pandas", pd.__version__)
if pd.__version__ != "3.0.6":
    print("Aviso: esta actividad se ha verificado con pandas 3.0.6; con otras versiones algunos resultados pueden diferir.")

# Los '..' del origen se leen como «sin dato» y, solo para poder sumar entrante + saliente, se cuentan como 0.
df = pd.read_csv(RUTA, na_values=[".."], keep_default_na=False, parse_dates=["Month"])
columnas_origen = df.shape[1]
df["Pasajeros"] = df[["Passengers In", "Passengers Out"]].fillna(0).sum(axis=1)       # pasajeros entrantes + salientes
df["Carga_t"] = df[["Freight In", "Freight Out"]].fillna(0).sum(axis=1)                 # carga entrante + saliente, en toneladas
print(f"Tabla: {len(df)} filas × {columnas_origen} columnas · {df['Month'].min():%Y-%m} → {df['Month'].max():%Y-%m}")
print("Columnas:", list(df.columns[:columnas_origen]))
repetidas = int(df.duplicated(["Month", "Scheduled Operator", "Country to/from"]).sum())
comprobar(repetidas == 0, "Cada fila es un mes × una aerolínea × un país del puerto (ninguna combinación se repite)")

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve y qué significa el resultado)

def serie(operador, medida, completa=False):
    """Serie mensual de una aerolínea (suma de todos sus países). Con completa=True, los meses sin filas aparecen con 0."""
    s = df[df["Scheduled Operator"] == operador].groupby("Month")[medida].sum()
    if completa and len(s):
        s = s.reindex(pd.date_range(s.index.min(), df["Month"].max(), freq="MS"), fill_value=0)
    return s


def meses_sin_filas(operador):
    """Meses entre el primer y el último mes de una aerolínea en los que no tiene ninguna fila."""
    meses = df.loc[df["Scheduled Operator"] == operador, "Month"].drop_duplicates()
    return int(len(pd.date_range(meses.min(), meses.max(), freq="MS")) - len(meses))


def ventana(s, mes, antes=3, despues=3):
    """Los meses de alrededor de «mes» (formato AAAA-MM) con su valor y su cambio frente al mes anterior."""
    if not (len(mes) == 7 and mes[4] == "-" and mes[:4].isdigit() and mes[5:].isdigit()):
        return None
    m = pd.Timestamp(mes + "-01")
    if m not in s.index:
        return None
    i = s.index.get_loc(m)
    w = s.iloc[max(0, i - antes): i + despues + 1]
    cambio = (s.pct_change() * 100).replace([float("inf"), float("-inf")], float("nan")).round(1).reindex(w.index)
    t = pd.DataFrame({"valor": w.round(0).astype(int), "cambio vs mes anterior (%)": cambio})
    t.index = t.index.strftime("%Y-%m")
    return t


def leer_extracto():
    """Diccionario ID → texto literal de cada bloque del extracto de documentación."""
    bloques, actual = {}, None
    for linea in open(RUTA_EXTRACTO, encoding="utf-8").read().splitlines():
        if linea.startswith("### "):
            actual = linea[4:].split(" ")[0]
            bloques[actual] = []
        elif linea.startswith(">") and actual:
            bloques[actual].append(linea[1:].strip())
    return {k: "\n".join(x for x in v if x) for k, v in bloques.items()}


EXTRACTO = leer_extracto()
print(f"Extracto de documentación: {len(EXTRACTO)} bloques con ID")

---
## Parte 1 · ENTENDER: las cinco afirmaciones y sus series (≈ 5,5 min)

Cada afirmación va con **el gráfico que vio la dirección**. Léelas con calma: son el material de toda la actividad.

### Afirmación 1
> «**Tasman Cargo Airlines ha perdido casi la mitad de su carga en Australia**: de enero a noviembre de 2025 transportó 35.413 t, frente a 62.718 t en el mismo periodo de 2024 (−43,5 %). Está perdiendo mercado.»

![Gráfico de líneas: carga mensual de Tasman Cargo Airlines en toneladas, de 2009 a 2025](https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/BigData/PA-06/data/figuras/afirmacion_1.png)

### Afirmación 2
> «**Singapur es nuestro segundo mercado internacional**, solo por detrás de Nueva Zelanda: 6,19 millones de pasajeros en 2024 (año natural), frente a 6,60 millones de Nueva Zelanda.»

![Gráfico de barras: pasajeros por país del puerto en 2024, los ocho primeros países](https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/BigData/PA-06/data/figuras/afirmacion_2.png)

### Afirmación 3
> «**Scoot desapareció del mercado australiano en agosto de 2017 y no volvió hasta agosto de 2025**: durante 96 meses no transportó ni un solo pasajero.»

![Gráfico de líneas: pasajeros mensuales de la aerolínea Scoot, de 2012 a 2025](https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/BigData/PA-06/data/figuras/afirmacion_3.png)

### Afirmación 4
> «**Vietnam Airlines aumentó sus pasajeros de marzo a abril de 2014**: de 24.977 a 25.885 (+3,6 %).»

![Gráfico de líneas: pasajeros mensuales de Vietnam Airlines, de 2009 a 2025](https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/BigData/PA-06/data/figuras/afirmacion_4.png)

### Afirmación 5
> «**Los pasajeros internacionales se han duplicado entre 2009 y 2025**: de 22,0 millones a 44,5 millones (×2,02).»

![Gráfico de barras: pasajeros internacionales por año según la columna Year, de 2009 a 2025](https://raw.githubusercontent.com/ericvd-ucb/universidad-europea-collab/main/BigData/PA-06/data/figuras/afirmacion_5.png)

### 1.1 ¿Las cifras están bien calculadas?

Antes de dudar de nada, comprueba que cada cifra de las afirmaciones **sale de la tabla**.

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve y qué significa el resultado)

t1 = serie("Tasman Cargo Airlines", "Carga_t")
a25, a24 = t1["2025-01":"2025-11"].sum(), t1["2024-01":"2024-11"].sum()
comprobar(round(a25) == 35413 and round(a24) == 62718 and round((a25 / a24 - 1) * 100, 1) == -43.5, f"Afirmación 1: enero–noviembre de 2025 = {a25:,.0f} t; 2024 = {a24:,.0f} t; variación {(a25 / a24 - 1) * 100:+.1f} %")

por_pais = df[df["Month"].dt.year == 2024].groupby("Country to/from")["Pasajeros"].sum().sort_values(ascending=False)
comprobar(por_pais.index[1] == "Singapore" and round(por_pais["Singapore"] / 1e6, 2) == 6.19 and round(por_pais["New Zealand"] / 1e6, 2) == 6.60, f"Afirmación 2: Singapur {por_pais['Singapore'] / 1e6:.2f} M, segundo; Nueva Zelanda {por_pais['New Zealand'] / 1e6:.2f} M")

hueco = meses_sin_filas("Scoot")
comprobar(hueco == 96, f"Afirmación 3: la aerolínea «Scoot» no tiene ninguna fila durante {hueco} meses entre su primer y su último mes")

v = serie("Vietnam Airlines", "Pasajeros")
marzo, abril = v["2014-03-01"], v["2014-04-01"]
comprobar(round(marzo) == 24977 and round(abril) == 25885 and round((abril / marzo - 1) * 100, 1) == 3.6, f"Afirmación 4: marzo de 2014 = {marzo:,.0f}; abril = {abril:,.0f}; variación {(abril / marzo - 1) * 100:+.1f} %")

anual = df.groupby("Year")["Pasajeros"].sum()
comprobar(round(anual[2009] / 1e6, 1) == 22.0 and round(anual[2025] / 1e6, 1) == 44.5 and round(anual[2025] / anual[2009], 2) == 2.02, f"Afirmación 5: Year 2009 = {anual[2009] / 1e6:.1f} M; Year 2025 = {anual[2025] / 1e6:.1f} M; razón ×{anual[2025] / anual[2009]:.2f}")
print()
print("Las cinco cifras salen de la tabla: la aritmética es correcta. La pregunta es otra: ¿se puede mantener cada afirmación?")

### 1.2 Tu primera impresión (solo con lo que ves)

**Sin leer todavía ninguna documentación**, escribe qué etiqueta pondrías a cada afirmación mirando solo la cifra y el gráfico. Es una declaración: no se evalúa, se contrasta al final. Valores posibles: `"ACCEPT"`, `"QUALIFY"`, `"RECOMPUTE"`, `"DO NOT COMPARE"`.

In [ ]:
primera_impresion = {1: "", 2: "", 3: "", 4: "", 5: ""}

---
## Parte 2 · ENTENDER: dónde cambia cada serie (≈ 1,5 min)

Mira los gráficos otra vez. Para cada afirmación, escribe **el mes** (formato `"AAAA-MM"`) en el que **tú** ves que la serie cambia. Si no ves ningún cambio, o el gráfico no es una serie mensual, escribe `"ninguno"`.

In [ ]:
mes_del_cambio = {1: "", 2: "", 3: "", 4: "", 5: ""}

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve y qué significa el resultado)

SERIES_A_MIRAR = {1: ("Tasman Cargo Airlines", "Carga_t", False), 3: ("Scoot", "Pasajeros", True), 4: ("Vietnam Airlines", "Pasajeros", False)}
for n in range(1, 6):
    mes = mes_del_cambio.get(n, "")
    print(f"--- Afirmación {n}")
    if not mes:
        print("✘ (pendiente) Escribe un mes o «ninguno» en la celda anterior.")
    elif mes == "ninguno":
        print("Has indicado que no ves ningún cambio mensual.")
    elif n not in SERIES_A_MIRAR:
        print("El gráfico de esta afirmación no es una serie mensual por aerolínea: no hay ventana de meses que mostrar.")
    else:
        operador, medida, completa = SERIES_A_MIRAR[n]
        w = ventana(serie(operador, medida, completa), mes)
        if w is None:
            print(f"✘ «{mes}» no es un mes de esta serie (usa el formato AAAA-MM).")
        else:
            print(f"{operador} ({medida})")
            print(w.to_string())

---
## Parte 3 · ENTENDER: la documentación (≈ 4,5 min)

Lee el **extracto de la documentación del libro de origen**. Son copias literales, en inglés. Cada bloque tiene un **ID** (por ejemplo `EN-14` o `DI-1`) que citarás como evidencia.

Mientras lees, ten presente la pregunta de toda la actividad: **¿lo que dice cada línea afecta a lo que se compara en esta tabla?** Una incidencia documentada puede afectar a otra cosa (a otra aerolínea, a otro periodo, a otra unidad).

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve y qué significa el resultado)

display(Markdown(open(RUTA_EXTRACTO, encoding="utf-8").read()))

---
## Parte 4 · DECIDIR: etiqueta, evidencia y defensa (≈ 7,5 min)

### 4.1 Una columna que hay que leer bien

La columna `Year` aparece en la tabla y en la afirmación 5. Antes de decidir, declara **qué crees que significa** (según la documentación). Valores posibles: `"año natural"`, `"año terminado en un mes concreto"`, `"no lo sé"`.

In [ ]:
mi_lectura_de_year = ""

### 4.2 Tu decisión sobre cada afirmación

Para **cada** afirmación escribe:

- `etiqueta`: una de las cuatro (`"ACCEPT"`, `"QUALIFY"`, `"RECOMPUTE"`, `"DO NOT COMPARE"`).
- `evidencia`: el **ID o los IDs** del extracto en los que te apoyas, separados por comas (por ejemplo `"EN-14, DI-1"`). Si crees que **ninguna** línea afecta, escribe `"ninguna"`… pero entonces explícalo en la defensa.
- `unidad_documentada`: **a qué unidad se refiere la línea que citas**: `"operador"`, `"país del puerto"`, `"ciudad"`, `"aeropuerto"`, `"mes o año"` o `"ninguna"`.
- `defensa`: **una frase** que diga por qué esa etiqueta y no otra.

In [ ]:
decision = {
    1: {"etiqueta": "", "evidencia": "", "unidad_documentada": "", "defensa": ""},
    2: {"etiqueta": "", "evidencia": "", "unidad_documentada": "", "defensa": ""},
    3: {"etiqueta": "", "evidencia": "", "unidad_documentada": "", "defensa": ""},
    4: {"etiqueta": "", "evidencia": "", "unidad_documentada": "", "defensa": ""},
    5: {"etiqueta": "", "evidencia": "", "unidad_documentada": "", "defensa": ""},
}

---
## Parte 5 · IMPLEMENTAR y VERIFICAR (≈ 2,5 min)

### 5.1 ¿Qué dicen las líneas que has citado y qué unidades tiene la tabla?

El código te enseña, para cada afirmación, **el texto literal** de lo que citaste y si **la unidad que indicaste existe en la tabla**.

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve y qué significa el resultado)

UNIDADES_DE_LA_TABLA = {"operador": "Scheduled Operator", "país del puerto": "Country to/from", "mes o año": "Month / Year"}


def ids_citados(texto):
    return [x.strip() for x in texto.replace(";", ",").split(",") if x.strip() and x.strip().lower() != "ninguna"]


if not all(decision[n]["etiqueta"] for n in decision):
    print("✘ (pendiente) Completa la decisión de la parte 4.2 para ver esta comprobación.")
else:
    print("Columnas de la tabla:", ", ".join(f"{u} ({c})" for u, c in UNIDADES_DE_LA_TABLA.items()), "· no hay ciudades ni aeropuertos")
    for n, d in decision.items():
        print(f"\n=== Afirmación {n} · etiqueta «{d['etiqueta']}»")
        comprobar(d["etiqueta"] in ETIQUETAS, "la etiqueta es una de las cuatro")
        citados = ids_citados(d["evidencia"])
        if not citados:
            print("— No citas ninguna línea (debes justificarlo en la defensa).")
        for i in citados:
            if i in EXTRACTO:
                comprobar(True, f"el ID {i} existe en el extracto. Texto literal: «{EXTRACTO[i]}»")
            else:
                comprobar(False, f"el ID {i} NO existe en el extracto")
        u = d["unidad_documentada"]
        if u in UNIDADES_DE_LA_TABLA:
            print(f"   La unidad que indicas («{u}») SÍ existe en la tabla: columna {UNIDADES_DE_LA_TABLA[u]}.")
        elif u in ("ciudad", "aeropuerto"):
            print(f"   La unidad que indicas («{u}») NO existe en la tabla: no hay ninguna columna por {u}.")
        elif u == "ninguna":
            print("   Indicas que ninguna unidad documentada interviene.")
        else:
            print("✘ unidad no válida; usa «operador», «país del puerto», «ciudad», «aeropuerto», «mes o año» o «ninguna».")

> **Pregunta silenciosa.** ¿Alguna de las líneas que has citado habla de algo que **no está en esta tabla**? ¿Cuál de tus afirmaciones depende de ello?

### 5.2 ¿Qué significa `Year`?

Contrasta tu declaración de 4.1 con la tabla: para algunos años, el primer y el último mes que contienen y cuántos meses son.

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve y qué significa el resultado)

cobertura = df.groupby("Year")["Month"].agg(primero="min", ultimo="max", meses="nunique")
cobertura["primero"], cobertura["ultimo"] = cobertura["primero"].dt.strftime("%Y-%m"), cobertura["ultimo"].dt.strftime("%Y-%m")
print(cobertura.loc[[2009, 2010, 2024, 2025]].to_string())
terminan_en = {int(m[-2:]) for m in cobertura["ultimo"].iloc[:-1]}
observado = "año terminado en un mes concreto" if len(terminan_en) == 1 and terminan_en != {12} else "año natural"
print(f"Mes en el que terminan los años 2009–2024: {sorted(terminan_en)}")
if mi_lectura_de_year:
    comprobar(mi_lectura_de_year == observado, f"Declaraste «{mi_lectura_de_year}»; lo que muestra la tabla es «{observado}»")
else:
    print("✘ (pendiente) Declara primero tu lectura de Year en 4.1.")

### 5.3 Dos herramientas de recálculo (úsalas solo si tu decisión las necesita)

Las dos están disponibles **para cualquier afirmación**: tú decides si alguna de tus decisiones las requiere y con qué parámetros. Si no las necesitas, deja las celdas en blanco.

**Herramienta A · unir denominaciones de una aerolínea.** Escribe un diccionario `{"nombre que se sustituye": "nombre que se queda"}` con la regla que **tú** apoyas en la documentación. El código aplica tu regla y, para cada nombre resultante, cuenta los meses sin filas entre su primer y su último mes, antes y después de unir.

In [ ]:
alias_operadores = {}

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve y qué significa el resultado)

if not alias_operadores:
    print("✘ (pendiente) Herramienta A sin usar: escribe una regla en la celda anterior si tu decisión la necesita.")
else:
    d2 = df.copy()
    d2["Operador_unificado"] = d2["Scheduled Operator"].replace(alias_operadores)
    for nombre in sorted(set(alias_operadores.values())):
        original = df.loc[df["Scheduled Operator"] == nombre, "Month"].drop_duplicates()
        if original.empty:
            print(f"✘ «{nombre}» no es el nombre de ninguna aerolínea de la tabla.")
            continue
        s_unif = d2[d2["Operador_unificado"] == nombre].groupby("Month")["Pasajeros"].sum()
        sin_filas = len(pd.date_range(s_unif.index.min(), s_unif.index.max(), freq="MS")) - len(s_unif)
        comprobar(sin_filas == 0, f"«{nombre}»: tras tu regla, meses sin filas entre su primer y último mes = {sin_filas} (antes de unir: {meses_sin_filas(nombre)})")
        faltan = pd.date_range(original.min(), original.max(), freq="MS").difference(original)
        if len(faltan):
            s_full = s_unif.reindex(pd.date_range(s_unif.index.min(), s_unif.index.max(), freq="MS"), fill_value=0)
            print(f"--- «{nombre}» tras tu regla, alrededor de su primer mes sin filas original ({faltan[0]:%Y-%m})")
            print(ventana(s_full, f"{faltan[0]:%Y-%m}").to_string())

**Herramienta B · comparar los mismos meses en varios años.** Escribe en `años` los años y en `meses` los **números de mes** (1 = enero … 12 = diciembre) que quieres comparar. El código suma los pasajeros de todas las aerolíneas de esos meses en cada año, te dice cuántos de esos meses existen en la tabla y calcula la razón entre el último año y el primero.

In [ ]:
comparar_meses = {"años": [], "meses": []}

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve y qué significa el resultado)

if not comparar_meses["años"] or not comparar_meses["meses"]:
    print("✘ (pendiente) Herramienta B sin usar: escribe años y meses en la celda anterior si tu decisión la necesita.")
elif len(comparar_meses["años"]) < 2:
    print("✘ Escribe al menos dos años.")
else:
    cifras = {}
    for año in comparar_meses["años"]:
        d = df[(df["Month"].dt.year == año) & (df["Month"].dt.month.isin(comparar_meses["meses"]))]
        cifras[año] = (d["Pasajeros"].sum(), d["Month"].nunique())
        print(f"{año}: {cifras[año][1]} de los {len(set(comparar_meses['meses']))} meses elegidos existen en la tabla · {cifras[año][0]:,.0f} pasajeros")
    comprobar(all(c[1] == len(set(comparar_meses["meses"])) for c in cifras.values()), "todos los años cubren exactamente los mismos meses que has elegido")
    primero, ultimo = comparar_meses["años"][0], comparar_meses["años"][-1]
    print(f"Razón {ultimo} / {primero} con tus meses: ×{cifras[ultimo][0] / cifras[primero][0]:.3f}")
    if primero in anual.index and ultimo in anual.index:
        print(f"Razón {ultimo} / {primero} con los totales de la columna Year: ×{anual[ultimo] / anual[primero]:.3f}")

---
## Parte 6 · DEFENDER: tu balance (≈ 0,5 min)

El código compara tu primera impresión con tu etiqueta final.

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve y qué significa el resultado)

if not all(decision[n]["etiqueta"] for n in decision) or not all(primera_impresion.values()):
    print("✘ (pendiente) Necesitas tus primeras impresiones (1.2) y tus etiquetas (4.2).")
else:
    filas = [(n, primera_impresion[n], decision[n]["etiqueta"], "cambió" if primera_impresion[n] != decision[n]["etiqueta"] else "igual") for n in decision]
    print(pd.DataFrame(filas, columns=["afirmación", "primera impresión", "etiqueta final", ""]).to_string(index=False))
    print(f"\nHas cambiado {sum(1 for f in filas if f[3] == 'cambió')} de {len(filas)} etiquetas después de leer la documentación y comprobar la tabla.")

In [ ]:
mi_balance = ""     # una frase: ¿qué te ha hecho cambiar (o no) de etiqueta en la afirmación en la que más has dudado?

In [ ]:
print("Estado de tu entrega (parte 0 a 6):")
print(" · primeras impresiones:", all(bool(v) for v in primera_impresion.values()))
print(" · meses del cambio:", all(bool(v) for v in mes_del_cambio.values()))
print(" · lectura de Year:", bool(mi_lectura_de_year))
print(" · decisiones completas:", all(all(bool(x) for x in d.values()) for d in decision.values()))
print(" · balance escrito:", bool(mi_balance))

---
# ⏸ PARADA DEL NÚCLEO DE CLASE

**Deja de escribir.** Lo que queda (parte 7) es para casa. Ahora toca la **puesta en común** (≈ 5 min). Ten a mano:

1. Qué etiqueta pusiste a cada afirmación **antes** y **después** de leer la documentación.
2. Qué línea citaste en cada una y **si la línea afecta a lo que se compara en esta tabla**.
3. Qué afirmaciones se pueden recalcular y con qué regla documentada, y cuáles no.
4. Una afirmación que **mantuvieras tal cual** y por qué.

---
## Parte 7 · Para casa (≤ 20 min): afina y defiende

**Qué no se hace aquí:** ningún concepto nuevo. Es dejar por escrito lo razonado en clase, auditar una propuesta y, si quieres, mirar un detalle de las fechas.

### 7.1 ¿Qué cambiaría tu respuesta?

Para cada afirmación, **una frase corta**: ¿qué dato, qué línea documentada o qué comprobación te haría cambiar de etiqueta?

In [ ]:
que_cambiaria_mi_respuesta = {1: "", 2: "", 3: "", 4: "", 5: ""}

### 7.2 Una propuesta de un asistente (para auditar)

Un asistente de IA propone esto para la afirmación 1. **No está verificada.** Léela y decide.

> «La caída de Tasman Cargo en enero de 2025 se debe a que antes se medía el peso bruto y después el neto. El peso neto suele ser aproximadamente el 80 % del bruto, así que multiplico por 0,80 todas las toneladas de enero–noviembre de 2024 y las comparo con 2025: 62.718 t × 0,80 = 50.174 t frente a 35.413 t, es decir, −29,4 %. Conclusión: Tasman sigue perdiendo carga, pero la caída real es del −29 %, no del −43 %.»

Escribe `veredicto` (`"la acepto"`, `"la acepto con cambios"` o `"la rechazo"`) y `razon`: **qué partes de la propuesta apoya la documentación y cuáles no**, con los IDs del extracto.

In [ ]:
auditoria_asistente = {"veredicto": "", "razon": ""}

### 7.3 Tu memo sobre la afirmación más difícil

Elige **la afirmación en la que más has dudado** (escribe su número en `afirmacion`) y redacta en `texto` un memo de **5 a 8 líneas** para la dirección: qué se afirmaba, qué etiqueta le pones, qué línea documentada lo sostiene, **si esa línea afecta a esta tabla** y cómo debería reformularse (o por qué no se puede).

In [ ]:
memo = {"afirmacion": None, "texto": ""}

### 7.4 (opcional, no se entrega) ¿Coincide la fecha documentada con la que ves en los datos?

Elige **una afirmación en la que hayas citado una línea con una fecha** y compara esa fecha con el mes en el que **tú** ves el cambio en los datos (parte 2). Escribe su número en `afirmacion_con_fecha` y, en `nota_fechas`, qué observas: ¿coinciden?, ¿qué haces si no?

In [ ]:
nota_fechas = {"afirmacion_con_fecha": None, "nota": ""}

In [ ]:
# CÓDIGO PROPORCIONADO — NO SE EVALÚA ESTA SINTAXIS (tú respondes de qué hace, qué devuelve y qué significa el resultado)

n = nota_fechas["afirmacion_con_fecha"]
if n not in decision or not decision[n]["evidencia"]:
    print("✘ (pendiente) Elige una afirmación en la que hayas citado una línea (opcional).")
else:
    print(f"Tu mes del cambio (parte 2) para la afirmación {n}: «{mes_del_cambio[n]}»")
    for i in ids_citados(decision[n]["evidencia"]):
        print(f"Línea que citaste ({i}):", EXTRACTO.get(i, "(ID no encontrado)"))

In [ ]:
print("Estado de tu entrega (casa):")
print(" · qué cambiaría mi respuesta:", all(bool(v) for v in que_cambiaria_mi_respuesta.values()))
print(" · auditoría del asistente:", all(bool(v) for v in auditoria_asistente.values()))
print(" · memo:", memo["afirmacion"] is not None and bool(memo["texto"]))

### Lista de verificación antes de entregar
- [ ] Declaré mi primera impresión **antes** de leer la documentación.
- [ ] En cada afirmación cité una línea concreta (o dije por qué ninguna aplica) y comprobé **qué unidad** tiene esa línea frente a la tabla.
- [ ] Solo recalculé donde la documentación daba una regla; en el resto no inventé factores.
- [ ] Sé decir para cada afirmación **qué lado de la comparación** está en qué base.
- [ ] Si he usado IA o documentación externa, entiendo lo que he copiado, lo he comprobado sobre estos datos y he dicho qué parte de mi conclusión depende de ella.

### Para quien quiera más (opcional)
- Busca en el extracto una línea que **no** afecte a ninguna de las cinco afirmaciones y explica por qué.
- Escribe una sexta afirmación cuyo cálculo sea correcto y cuya etiqueta correcta sea `ACCEPT`, apoyándote en una línea documentada que no aplique.